# Movie Rating Analysis

This notebook adapts the existing `analysis.py` workflow for interactive use. It reads the MovieLens CSV files, computes movie, genre, rating-distribution, and release-year summaries with PySpark, saves CSV tables and charts under `output/`, and can optionally load the complete raw ratings dataset plus summaries into Cassandra.

Run the notebook from the `movie-rating-analysis` project folder. The notebook and `analysis.py` are separate entry points; each can load the full ratings dataset into Cassandra.


## 1. Imports and project settings

The notebook uses the project directory as its base so it does not depend on Python's `__file__`, which is not defined in notebooks. The input files are expected at `data/movie.csv` and `data/rating.csv`.

In [1]:
import csv
import logging
import os
from pathlib import Path
from typing import Iterable, Mapping

from pyspark.sql import DataFrame, SparkSession, functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
TABLE_DIR = OUTPUT_DIR / "tables"
CHART_DIR = OUTPUT_DIR / "charts"
KEYSPACE = "movie_analysis"

for required_file in (DATA_DIR / "movie.csv", DATA_DIR / "rating.csv"):
    if not required_file.is_file():
        raise FileNotFoundError(
            f"Required file not found: {required_file}. Run this notebook from the project folder."
        )

MOVIE_SCHEMA = StructType(
    [
        StructField("movieId", IntegerType(), nullable=False),
        StructField("title", StringType(), nullable=True),
        StructField("genres", StringType(), nullable=True),
    ]
)
RATING_SCHEMA = StructType(
    [
        StructField("userId", IntegerType(), nullable=False),
        StructField("movieId", IntegerType(), nullable=False),
        StructField("rating", DoubleType(), nullable=False),
        StructField("timestamp", StringType(), nullable=True),
    ]
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
print(f"Project folder: {BASE_DIR}")

Project folder: /Users/krrishparmar/movie-rating-analysis


## 2. Start Spark



In [2]:
spark = (
    SparkSession.builder.appName("MovieRatingAnalysis")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/27 15:47:34 WARN Utils: Your hostname, Krrishs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 172.16.212.184 instead (on interface en0)
26/10/27 15:47:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/27 15:47:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 3. Load the MovieLens data

Both files are read with explicit schemas, as in the script. The timestamp is retained as text because MovieLens releases use more than one timestamp format.

In [3]:
movies = spark.read.option("header", True).schema(MOVIE_SCHEMA).csv(str(DATA_DIR / "movie.csv"))
ratings = spark.read.option("header", True).schema(RATING_SCHEMA).csv(str(DATA_DIR / "rating.csv"))

source_rating_count = ratings.count()
print(f"Movies: {movies.count():,}")
print(f"Ratings: {source_rating_count:,}")


Movies: 27,278
Ratings: 20,000,263


## 4. Build the analysis aggregations

The four summaries retain the original analysis logic: one row per movie, genre-level averages and counts, counts for each rating value, and averages and counts by parsed movie release year.

In [4]:
movie_stats = (
    movies.join(
        ratings.groupBy("movieId").agg(
            F.avg("rating").alias("average_rating"),
            F.count("rating").alias("rating_count"),
        ),
        "movieId",
        "left",
    )
    .select(
        F.col("movieId").alias("movie_id"),
        "title",
        "genres",
        F.round("average_rating", 4).alias("average_rating"),
        F.coalesce("rating_count", F.lit(0)).cast("long").alias("rating_count"),
    )
)

movie_genres = movies.select(
    "movieId", F.explode(F.split(F.col("genres"), "\\|")).alias("genre")
)
genre_stats = (
    movie_genres.join(ratings, "movieId")
    .where(F.col("genre") != "(no genres listed)")
    .groupBy("genre")
    .agg(
        F.round(F.avg("rating"), 4).alias("average_rating"),
        F.count("rating").alias("rating_count"),
    )
)

rating_distribution = ratings.groupBy("rating").agg(
    F.count("rating").alias("rating_count")
)

release_year = F.regexp_extract("title", r"\((\d{4})\)\s*$", 1)
movie_years = (
    movies.withColumn(
        "year", F.when(release_year != "", release_year.cast("int"))
    )
    .where(F.col("year").isNotNull() & (F.col("year") > 0))
)
yearly_stats = (
    movie_years.join(ratings, "movieId")
    .groupBy("year")
    .agg(
        F.round(F.avg("rating"), 4).alias("average_rating"),
        F.count("rating").alias("rating_count"),
    )
)

summaries = {
    "movie_stats": movie_stats,
    "genre_stats": genre_stats,
    "rating_distribution": rating_distribution,
    "yearly_stats": yearly_stats,
}

for name, frame in summaries.items():
    print(f"{name}: {frame.count():,} rows")

movie_stats: 27,278 rows


genre_stats: 19 rows


rating_distribution: 10 rows


yearly_stats: 118 rows


## 5. Collect and save summary tables

The summary rows are collected to the driver, as in the original script, then written as CSV files under `output/tables/`.

In [5]:
rows = {
    name: [row.asDict(recursive=True) for row in frame.collect()]
    for name, frame in summaries.items()
}

def write_csv(data: Iterable[Mapping[str, object]], path: Path, columns: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(data)

for name, records in rows.items():
    columns = list(records[0].keys()) if records else []
    output_path = TABLE_DIR / f"{name}.csv"
    write_csv(records, output_path, columns)
    print(f"Wrote {len(records):,} rows to {output_path.relative_to(BASE_DIR)}")

Wrote 27,278 rows to output/tables/movie_stats.csv
Wrote 19 rows to output/tables/genre_stats.csv
Wrote 10 rows to output/tables/rating_distribution.csv
Wrote 118 rows to output/tables/yearly_stats.csv


## 6. Generate charts

This cell creates the same three PNG charts as `analysis.py`.

In [6]:
CHART_DIR.mkdir(parents=True, exist_ok=True)
matplotlib_config = OUTPUT_DIR / ".matplotlib"
matplotlib_config.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_config))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

distribution = sorted(rows["rating_distribution"], key=lambda row: float(row["rating"]))
plt.figure(figsize=(8, 5))
plt.bar([str(row["rating"]) for row in distribution], [int(row["rating_count"]) for row in distribution])
plt.title("MovieLens Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of ratings")
plt.tight_layout()
plt.savefig(CHART_DIR / "rating_distribution.png", dpi=160)
plt.close()

genres = sorted(rows["genre_stats"], key=lambda row: int(row["rating_count"]), reverse=True)[:12]
genres.reverse()
plt.figure(figsize=(9, 6))
plt.barh([str(row["genre"]) for row in genres], [int(row["rating_count"]) for row in genres])
plt.title("Most Rated Genres")
plt.xlabel("Number of ratings")
plt.tight_layout()
plt.savefig(CHART_DIR / "genre_rating_counts.png", dpi=160)
plt.close()

yearly = sorted(rows["yearly_stats"], key=lambda row: int(row["year"]))
plt.figure(figsize=(10, 5))
plt.plot([int(row["year"]) for row in yearly], [float(row["average_rating"]) for row in yearly], marker="o", markersize=2)
plt.title("Average Rating by Movie Release Year")
plt.xlabel("Release year")
plt.ylabel("Average rating")
plt.tight_layout()
plt.savefig(CHART_DIR / "average_rating_by_year.png", dpi=160)
plt.close()

print(f"Charts saved to {CHART_DIR}")

2026-10-27 15:47:52,757 INFO Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-10-27 15:47:52,757 INFO Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Charts saved to /Users/krrishparmar/movie-rating-analysis/output/charts


## 7. Optional Cassandra loading

Cassandra loading is off by default. Set `LOAD_CASSANDRA = True` after starting Cassandra and applying `cassandra_schema.cql` to stream every raw rating record into `rating_records`, load the four summary tables, and compare stored counts with the source and generated row counts. The raw import is bounded and can be rerun safely because each user/movie pair is the table key.


In [ ]:
LOAD_CASSANDRA = True
CASSANDRA_HOST = "127.0.0.1"
CASSANDRA_PORT = 9042

if LOAD_CASSANDRA:
    from cassandra.cluster import Cluster
    from cassandra.concurrent import execute_concurrent_with_args

    cluster = Cluster([CASSANDRA_HOST], port=CASSANDRA_PORT, protocol_version=5)
    try:
        session = cluster.connect()
        raw_statement = session.prepare(
            f"INSERT INTO {KEYSPACE}.rating_records "
            "(user_id, movie_id, rating, rating_timestamp) VALUES (?, ?, ?, ?)"
        )

        def raw_rating_rows():
            with (DATA_DIR / "rating.csv").open(
                "r", newline="", encoding="utf-8"
            ) as handle:
                reader = csv.DictReader(handle)
                expected_columns = ["userId", "movieId", "rating", "timestamp"]
                if reader.fieldnames != expected_columns:
                    raise ValueError(
                        f"Unexpected ratings CSV columns: {reader.fieldnames!r}"
                    )
                for line_number, record in enumerate(reader, start=2):
                    try:
                        yield (
                            int(record["userId"]),
                            int(record["movieId"]),
                            float(record["rating"]),
                            record["timestamp"],
                        )
                    except (TypeError, ValueError) as exc:
                        raise ValueError(
                            f"Invalid ratings CSV record at line {line_number}: {record!r}"
                        ) from exc

        raw_results = execute_concurrent_with_args(
            session,
            raw_statement,
            raw_rating_rows(),
            concurrency=64,
            raise_on_first_error=True,
            results_generator=True,
        )
        loaded_rating_count = sum(1 for _success, _result in raw_results)
        if loaded_rating_count != source_rating_count:
            raise RuntimeError(
                f"Read {loaded_rating_count:,} raw ratings, expected {source_rating_count:,}"
            )
        logging.info("Loaded %s raw rows into rating_records", f"{loaded_rating_count:,}")

        statements = {
            "movie_stats": "INSERT INTO movie_stats (movie_id, title, genres, average_rating, rating_count) VALUES (?, ?, ?, ?, ?)",
            "genre_stats": "INSERT INTO genre_stats (genre, average_rating, rating_count) VALUES (?, ?, ?)",
            "rating_distribution": "INSERT INTO rating_distribution (rating, rating_count) VALUES (?, ?)",
            "yearly_stats": "INSERT INTO yearly_stats (year, average_rating, rating_count) VALUES (?, ?, ?)",
        }
        for table, cql in statements.items():
            prepared = session.prepare(
                cql.replace(f"INTO {table} ", f"INTO {KEYSPACE}.{table} ")
            )
            if table == "movie_stats":
                args = ((r["movie_id"], r["title"], r["genres"], r["average_rating"], r["rating_count"]) for r in rows[table])
            elif table == "genre_stats":
                args = ((r["genre"], r["average_rating"], r["rating_count"]) for r in rows[table])
            elif table == "rating_distribution":
                args = ((r["rating"], r["rating_count"]) for r in rows[table])
            else:
                args = ((r["year"], r["average_rating"], r["rating_count"]) for r in rows[table])
            results = execute_concurrent_with_args(
                session, prepared, args, concurrency=32, raise_on_first_error=True
            )
            logging.info("Loaded %s rows into %s", len(results), table)

        expected_counts = {table: len(records) for table, records in rows.items()}
        expected_counts["rating_records"] = source_rating_count
        for table, expected_count in expected_counts.items():
            result = session.execute(
                f"SELECT COUNT(*) FROM {KEYSPACE}.{table}"
            ).one()
            stored_count = result.count if result else 0
            print(f"{table}: {stored_count:,} stored; {expected_count:,} expected")
            if stored_count != expected_count:
                raise RuntimeError(
                    f"Row-count mismatch for {table}: expected {expected_count:,}, found {stored_count:,}"
                )
    finally:
        cluster.shutdown()
else:
    print("Cassandra loading skipped. Set LOAD_CASSANDRA = True to enable it.")


2026-10-27 15:48:53,772 WARNING Cluster.__init__ called with contact_points specified, but no load_balancing_policy. In the next major version, this will raise an error; please specify a load-balancing policy. (contact_points = ['127.0.0.1'], lbp = None)
2026-10-27 15:48:53,854 INFO Using datacenter 'datacenter1' for DCAwareRoundRobinPolicy (via host '127.0.0.1:9042'); if incorrect, please specify a local_dc to the constructor, or limit contact points to local cluster nodes
2026-10-27 16:09:24,430 INFO Loaded 20,000,263 raw rows into rating_records
2026-10-27 16:09:26,304 INFO Loaded 27278 rows into movie_stats
2026-10-27 16:09:26,310 INFO Loaded 19 rows into genre_stats
2026-10-27 16:09:26,312 INFO Loaded 10 rows into rating_distribution
2026-10-27 16:09:26,319 INFO Loaded 118 rows into yearly_stats
2026-10-27 16:09:26,671 WARNING Server warning: Aggregation query used without partition key
2026-10-27 16:09:26,678 WARNING Server warning: Aggregation query used without partition key
20

movie_stats: 27,278 stored; 27,278 expected
genre_stats: 19 stored; 19 expected
rating_distribution: 10 stored; 10 expected
yearly_stats: 118 stored; 118 expected


OperationTimedOut: errors={'127.0.0.1:9042': 'Client request timeout. See Session.execute[_async](timeout)'}, last_host=127.0.0.1:9042

26/10/27 16:26:06 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 454899 ms exceeds timeout 120000 ms
26/10/27 16:26:06 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/27 16:27:39 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

## 8. Stop Spark

Run this cell when you are finished with the analysis session.

In [ ]:
spark.stop()

## 9. Interactive Streamlit Dashboard Frontend (TMDB Movies Power BI UI)

This section builds the interactive Streamlit presentation dashboard matching the Power BI **Movie Industry Insights** UI.
It includes:
- **Overview Tab:** 6 interactive cards (Total Movies KPI, Movies Released Across Runtime, Distribution by Language with Drill-down, Seasonal Trend by Decades/Years, Top 5 Production Companies, Top 5 Movie Genres).
- **Revenue Tab:** Box office economics, Top 10 highest-grossing films, genre revenue breakdown, and studio rankings.
- **Interactive Filters:** Movie Genres, Movie Language, Production Company, Info popup, and one-click filter reset.
- **MovieLens 20M Integration:** Direct exploration of local PySpark summary tables.

Running the cell below writes the complete frontend application to  (and ).


In [ ]:
import shutil
shutil.copyfile('analysis.py', 'app.py')
print('✓ Frontend written to analysis.py and synced to app.py!')
print('To launch the dashboard, run in Terminal:')
print('    streamlit run app.py')


✓ Frontend written to analysis.py and synced to app.py!
To launch the dashboard, run in Terminal:
    streamlit run app.py
